In [24]:
import os
import numpy as np
import pandas as pd
from scipy.fft import rfft, rfftfreq
from scipy.signal import find_peaks
from sklearn.preprocessing import MinMaxScaler
import joblib

In [17]:
#Plan to get from Time Domain: RMS, Kurtosis, Peak amplitude, Crest Factor(Peak/RMS), Shape Factor (RMS/Mean Abs Value)
#Plan from frequency domain: Top 50 Frequencies, Magnitudes of those frequencies...
# mean freq mag, std freq mag, spectral centroid(mean freq/sum of magnitudes)

In [18]:
def process_folder(folder_path, set_id, bearing_col=0):
    # Only grab files (skipping system files like desktop.ini)
    files = sorted([f for f in os.listdir(folder_path) if f not in ['desktop.ini', '.DS_Store']])
    data_list = []
    
    print(f"Processing {len(files)} files from Set {set_id}...")

    for i, filename in enumerate(files):
        try:
            file_path = os.path.join(folder_path, filename)
            # NASA files use tabs; header=None because there are no column titles
            raw_data = pd.read_csv(file_path, sep='\t', header=None)
            signal = raw_data[bearing_col].values
            
            # --- 1. TIME DOMAIN ---
            rms = np.sqrt(np.mean(signal**2))
            kurtosis = pd.Series(signal).kurt()
            peak = np.max(np.abs(signal))
            crest_factor = peak / rms if rms != 0 else 0
            shape_factor = rms / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
            
            # --- 2. FREQUENCY DOMAIN ---
            fs = 20000
            yf = np.abs(rfft(signal))
            xf = rfftfreq(len(signal), 1/fs)
            
            # Global Spectral Stats
            spec_mean = np.mean(yf)
            spec_std = np.std(yf)
            spec_centroid = np.sum(xf * yf) / np.sum(yf) if np.sum(yf) != 0 else 0
            
            # Peak Finding (Top 50)
            peaks, _ = find_peaks(yf, distance=100)
            top_indices = peaks[np.argsort(yf[peaks])[-50:]][::-1]
            
            freqs = np.zeros(50)
            mags = np.zeros(50)
            freqs[:len(top_indices)] = xf[top_indices]
            mags[:len(top_indices)] = yf[top_indices]
            
            # --- 3. COMBINE ---
            # Order: ID info + 5 Time Stats + 50 Freqs + 50 Mags + 3 Global Stats
            row = [filename, set_id, rms, kurtosis, peak, crest_factor, shape_factor] + \
                  list(freqs) + list(mags) + \
                  [spec_mean, spec_std, spec_centroid]
            
            data_list.append(row)
            
            if i % 100 == 0:
                print(f"Processed {i}/{len(files)} files...")
                
        except Exception as e:
            print(f"Error in {filename}: {e}")
            
    return data_list

In [ ]:
folder_set1 = os.path.join('raw_data', '1st_test')
print("\n--- Starting Set 1 ---")
# Note: bearing_col=0 extracts the first sensor of the first bearing
data_set1 = process_folder(folder_set1, set_id=1, bearing_col=0)
df_set1 = pd.DataFrame(data_set1, columns=columns)
df_set1.to_csv('processed_features_set1.csv', index=False)
print("Finished Set 1!")

In [19]:
target_folder = os.path.join('raw_data', '2nd_test')

# Define all columns for the final DataFrame
columns = ['timestamp', 'set_id', 'rms', 'kurtosis', 'peak', 'crest_factor', 'shape_factor'] + \
          [f'f_{i}' for i in range(50)] + \
          [f'm_{i}' for i in range(50)] + \
          ['spec_mean', 'spec_std', 'spec_centroid']

# Run the process
processed_data = process_folder(target_folder, set_id=2)

# Convert to DataFrame and Save
if processed_data:
    final_df = pd.DataFrame(processed_data, columns=columns)
    final_df.to_csv('processed_features_set2.csv', index=False)
    print("\nDone! CSV saved with all features.")

Processing 984 files from Set 2...
Processed 0/984 files...
Processed 100/984 files...
Processed 200/984 files...
Processed 300/984 files...
Processed 400/984 files...
Processed 500/984 files...
Processed 600/984 files...
Processed 700/984 files...
Processed 800/984 files...
Processed 900/984 files...

Done! CSV saved with all features.


In [ ]:
# Execute the processing
all_data = process_set('raw_data/2nd_test', set_id=2)

# Save to CSV
df_master = pd.DataFrame(all_data, columns=columns)
df_master.to_csv('bearing_features_50peaks.csv', index=False)
print("Master file saved successfully!")

In [21]:
target_folder = os.path.join('raw_data', '3rd_test')

# Define all columns for the final DataFrame
columns = ['timestamp', 'set_id', 'rms', 'kurtosis', 'peak', 'crest_factor', 'shape_factor'] + \
          [f'f_{i}' for i in range(50)] + \
          [f'm_{i}' for i in range(50)] + \
          ['spec_mean', 'spec_std', 'spec_centroid']

# Run the process
processed_data = process_folder(target_folder, set_id=3)

# Convert to DataFrame and Save
if processed_data:
    final_df = pd.DataFrame(processed_data, columns=columns)
    final_df.to_csv('processed_features_set3.csv', index=False)
    print("\nDone! CSV saved with all features.")

Processing 6324 files from Set 3...
Processed 0/6324 files...
Processed 100/6324 files...
Processed 200/6324 files...
Processed 300/6324 files...
Processed 400/6324 files...
Processed 500/6324 files...
Processed 600/6324 files...
Processed 700/6324 files...
Processed 800/6324 files...
Processed 900/6324 files...
Processed 1000/6324 files...
Processed 1100/6324 files...
Processed 1200/6324 files...
Processed 1300/6324 files...
Processed 1400/6324 files...
Processed 1500/6324 files...
Processed 1600/6324 files...
Processed 1700/6324 files...
Processed 1800/6324 files...
Processed 1900/6324 files...
Processed 2000/6324 files...
Processed 2100/6324 files...
Processed 2200/6324 files...
Processed 2300/6324 files...
Processed 2400/6324 files...
Processed 2500/6324 files...
Processed 2600/6324 files...
Processed 2700/6324 files...
Processed 2800/6324 files...
Processed 2900/6324 files...
Processed 3000/6324 files...
Processed 3100/6324 files...
Processed 3200/6324 files...
Processed 3300/6324


--- Starting Set 1 ---
Processing 2156 files from Set 1...
Processed 0/2156 files...
Processed 100/2156 files...
Processed 200/2156 files...
Processed 300/2156 files...
Processed 400/2156 files...
Processed 500/2156 files...
Processed 600/2156 files...
Processed 700/2156 files...
Processed 800/2156 files...
Processed 900/2156 files...
Processed 1000/2156 files...
Processed 1100/2156 files...
Processed 1200/2156 files...
Processed 1300/2156 files...
Processed 1400/2156 files...
Processed 1500/2156 files...
Processed 1600/2156 files...
Processed 1700/2156 files...
Processed 1800/2156 files...
Processed 1900/2156 files...
Processed 2000/2156 files...
Processed 2100/2156 files...
Finished Set 1!


In [25]:
# 1. Load your processed data
df2 = pd.read_csv('processed_features_set2.csv')

# 2. Identify the columns to scale (exclude 'timestamp' and 'set_id')
features_to_scale = df2.columns.drop(['timestamp', 'set_id'])

# 3. Create the Scaler
scaler = MinMaxScaler()

# 4. FIT ONLY ON HEALTHY DATA (First 500 rows of Set 2)
# This teaches the scaler what "Normal" looks like
healthy_data = df2.loc[:500, features_to_scale]
scaler.fit(healthy_data)

# 5. TRANSFORM THE DATA
# Now we apply that "Healthy Scale" to everything
df2_scaled_values = scaler.transform(df2[features_to_scale])

# Create a new DataFrame with the scaled values
df2_scaled = pd.DataFrame(df2_scaled_values, columns=features_to_scale)
df2_scaled.insert(0, 'timestamp', df2['timestamp']) # Put timestamp back

# 6. Save the Scaled CSV
df2_scaled.to_csv('scaled_features_set2.csv', index=False)

# 7. SAVE THE SCALER OBJECT
# Give this file to your partners! They need it to 'un-scale' 
# or process new data in the same way.
joblib.dump(scaler, 'bearing_scaler.pkl')

print("Normalization Complete. Scaled data saved to 'scaled_features_set2.csv'")

Normalization Complete. Scaled data saved to 'scaled_features_set2.csv'


In [ ]:
# To normalize Set 1:
#Note: This used different machines, so the scaled values are off. I scaled it based on set2 data
df1 = pd.read_csv('processed_features_set1.csv')
df1_scaled_values = scaler.transform(df1[features_to_scale])
# Create a new DataFrame with the scaled values
df1_scaled = pd.DataFrame(df1_scaled_values, columns=features_to_scale)
df1_scaled.insert(0, 'timestamp', df1['timestamp']) # Put timestamp back

# 6. Save the Scaled CSV
df1_scaled.to_csv('scaled_features_set1.csv', index=False)

In [27]:
# To normalize Set 3:
df3 = pd.read_csv('processed_features_set3.csv')
df3_scaled_values = scaler.transform(df3[features_to_scale])
# Create a new DataFrame with the scaled values
df3_scaled = pd.DataFrame(df3_scaled_values, columns=features_to_scale)
df3_scaled.insert(0, 'timestamp', df3['timestamp']) # Put timestamp back

# 6. Save the Scaled CSV
df3_scaled.to_csv('scaled_features_set3.csv', index=False)